# UK Smart Grid Forecaster — Updated Prophet Training

This notebook is the Prophet training workflow.

### Evaluation rules used here
- **Development data:** 2010-01-01 to 2026-05-31
- **Final untouched test:** June 2026
- Feature analysis, feature screening, cross-validation, and hyperparameter tuning use **development data only**
- Prophet UK holidays are added **only when explicitly requested by a model variant**
- `demand_lag_24` and `demand_lag_168` are evaluated because the target is a rolling **next-24-hour** electricity-demand forecast
- June is evaluated **once only after model selection**
- The production model may then be retrained using all available data

> Run the notebook top-to-bottom. Keep `RUN_FINAL_JUNE_TEST = False` until Prophet has been compared with the other models.


In [ ]:
import json
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
from prophet import Prophet
from prophet.serialize import model_to_json

pd.set_option('display.max_columns', 120)

INPUT_PATH = '/kaggle/input/datasets/kusalnirukshan/master-train-data-uk-demand/master_training_data.csv'

OUTPUT_FOLDER = (
    Path('/kaggle/working/prophet_outputs')
    if Path('/kaggle/working').exists()
    else Path('artifacts/prophet_notebook')
)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

INPUT_PATH, OUTPUT_FOLDER


## 1. Load and clean the dataset

Only forward filling is used for numeric regressors. Back-filling is intentionally avoided because it can move future information backward in time.


In [ ]:
df = pd.read_csv(INPUT_PATH, low_memory=False)

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')

df = (
    df
    .dropna(subset=['timestamp', 'demand_mw'])
    .sort_values('timestamp')
    .drop_duplicates(subset=['timestamp'], keep='last')
    .reset_index(drop=True)
)

print('Rows:', len(df))
print('Columns:', len(df.columns))
print('Range:', df['timestamp'].min(), 'to', df['timestamp'].max())
print('Duplicate timestamps:', df['timestamp'].duplicated().sum())

df.head()


In [ ]:
# Optional text-column cleanup.
# These columns are not used directly as Prophet regressors,
# but filling them makes later inspection easier.

text_cols = [
    'cal_holiday_names',
    'cal_holiday_regions',
    'holiday_name',
    'cal_event_names',
]

text_cols = [c for c in text_cols if c in df.columns]

if text_cols:
    df[text_cols] = df[text_cols].fillna('None')

missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(30)


## 2. Candidate regressors

The initial feature list contains numeric weather, economic, calendar, and event variables. All correlation-based decisions below are made using **development data only**.


In [ ]:
candidate_features = [
    # Weather
    'temperature_2m',
    'relative_humidity_2m',
    'dew_point_2m',
    'apparent_temperature',
    'precipitation',
    'rain',
    'surface_pressure',
    'cloud_cover',
    'wind_speed_10m',
    'wind_direction_10m',
    'shortwave_radiation',

    # Economy
    'econ_industrial_production_index_lag1m',
    'econ_gdp_index_lag1m',
    'econ_cpi_index_lag1m',
    'econ_unemployment_rate_lag1m',

    # Calendar / events
    'weekend',
    'is_holiday',
    'cal_is_bank_holiday',
    'cal_is_bank_holiday_england_wales',
    'cal_is_bank_holiday_scotland',
    'cal_event_count',
    'cal_is_covid_lockdown',
    'cal_is_general_election',
    'cal_is_major_football',
    'cal_is_event_day',
    'cal_is_non_working_day',
]

candidate_features = [c for c in candidate_features if c in df.columns]

# Forward-fill only. Do NOT bfill time-series regressors.
for col in candidate_features:
    df[col] = pd.to_numeric(df[col], errors='coerce').ffill()

candidate_features


## 3. Isolate the final June 2026 test set

The test set is created before feature selection, screening, or tuning. Historical demand lag features are also generated here.

The lag evaluation represents a **rolling 24-hour forecasting system**: at each forecast origin, demand from 24 and 168 hours earlier is already observed.


In [ ]:
FINAL_TEST_START = pd.Timestamp('2026-06-01 00:00:00')

# Verify that shift(24) and shift(168) genuinely correspond to 24/168 hours.
hour_diffs = df['timestamp'].diff().dropna()

assert (
    hour_diffs == pd.Timedelta(hours=1)
).all(), (
    'Dataset is not perfectly hourly. '
    'Check missing timestamps before using row-based demand lags.'
)

# Historical demand lags suitable for rolling next-24-hour prediction.
df['demand_lag_24'] = df['demand_mw'].shift(24)
df['demand_lag_168'] = df['demand_mw'].shift(168)

# Final holdout split.
dev_df = df[df['timestamp'] < FINAL_TEST_START].copy()
final_test_df = df[df['timestamp'] >= FINAL_TEST_START].copy()

print(
    'Development:',
    len(dev_df),
    dev_df['timestamp'].min(),
    'to',
    dev_df['timestamp'].max(),
)

print(
    'FINAL TEST:',
    len(final_test_df),
    final_test_df['timestamp'].min(),
    'to',
    final_test_df['timestamp'].max(),
)

assert dev_df['timestamp'].max() < FINAL_TEST_START
assert final_test_df['timestamp'].min() == FINAL_TEST_START
assert final_test_df['timestamp'].max() <= pd.Timestamp('2026-06-30 23:00:00')

print('\nJune 2026 is isolated and must remain untouched until final model selection.')


## 4. Development-only feature checks


In [ ]:
development_corr = (
    dev_df[candidate_features + ['demand_mw']]
    .corr(numeric_only=True)['demand_mw']
    .sort_values(key=abs, ascending=False)
)

development_corr


In [ ]:
# Check overlap among the different holiday indicators using DEVELOPMENT data only.

holiday_columns = [
    'is_holiday',
    'cal_is_bank_holiday_england_wales',
    'cal_is_bank_holiday_scotland',
    'cal_is_bank_holiday',
]

holiday_columns = [c for c in holiday_columns if c in dev_df.columns]

if 'is_holiday' in holiday_columns:
    for col in holiday_columns:
        if col != 'is_holiday':
            match_rate = (dev_df['is_holiday'] == dev_df[col]).mean()
            print(f'is_holiday == {col}: {match_rate:.4f}')


In [ ]:
# Remove duplicate / overlapping bank-holiday indicators.
# We keep is_holiday as the custom binary holiday candidate.

remove_features = [
    'cal_is_bank_holiday',
    'cal_is_bank_holiday_england_wales',
    'cal_is_bank_holiday_scotland',
]

candidate_features = [
    col for col in candidate_features
    if col not in remove_features
]

print(candidate_features)
print('Remaining candidate features:', len(candidate_features))


In [ ]:
# Multicollinearity check on DEVELOPMENT data only.

feature_corr = dev_df[candidate_features].corr(numeric_only=True)

upper = feature_corr.abs().where(
    np.triu(np.ones(feature_corr.shape), k=1).astype(bool)
)

high_corr_pairs = (
    upper
    .stack()
    .sort_values(ascending=False)
)

high_corr_pairs[high_corr_pairs > 0.80]


In [ ]:
# Preserve the feature-removal choices from the earlier analysis,
# but now make the workflow leakage-safe because the diagnostics above
# are based only on dev_df.

remove_features = [
    'cal_is_event_day',
    'rain',
    'temperature_2m',
    'dew_point_2m',
]

candidate_features = [
    col for col in candidate_features
    if col not in remove_features
]

print(candidate_features)
print('Final candidate count:', len(candidate_features))


In [ ]:
feature_report = []

for col in candidate_features:
    if dev_df[col].nunique(dropna=True) <= 1:
        continue

    feature_report.append({
        'feature': col,
        'unique_values': dev_df[col].nunique(dropna=True),
        'development_corr': dev_df[[col, 'demand_mw']].corr().iloc[0, 1],
        'missing_development': dev_df[col].isna().sum(),
    })

feature_report = pd.DataFrame(feature_report)

if not feature_report.empty:
    feature_report['abs_development_corr'] = (
        feature_report['development_corr'].abs()
    )

    feature_report = (
        feature_report
        .sort_values('abs_development_corr', ascending=False)
        .reset_index(drop=True)
    )

feature_report.to_csv(
    OUTPUT_FOLDER / 'feature_strength_development.csv',
    index=False,
)

feature_report


In [ ]:
# Optional diagnostic checks on development data only.

econ_cols = [
    'econ_industrial_production_index_lag1m',
    'econ_gdp_index_lag1m',
    'econ_cpi_index_lag1m',
    'econ_unemployment_rate_lag1m',
]
econ_cols = [c for c in econ_cols if c in dev_df.columns]

if econ_cols:
    print('Economic feature unique values:')
    display(dev_df[econ_cols].nunique())

if 'cal_is_covid_lockdown' in dev_df.columns:
    print('\nLockdown value counts:')
    display(dev_df['cal_is_covid_lockdown'].value_counts(dropna=False))


## 5. Prophet helpers

A key correction here is that `add_country_holidays('UK')` is **not** called automatically. A variant receives Prophet UK holidays only when `use_prophet_holidays=True`.


In [ ]:
def make_prophet_frame(source_df, regressors):
    required_columns = ['timestamp', 'demand_mw', *regressors]

    out = source_df[required_columns].copy()

    out = out.rename(
        columns={
            'timestamp': 'ds',
            'demand_mw': 'y',
        }
    )

    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=['y', *regressors])

    return out


def calculate_metrics(actual, predicted):
    actual = pd.Series(actual).reset_index(drop=True)
    predicted = pd.Series(predicted).reset_index(drop=True)

    error = actual - predicted

    mae = error.abs().mean()
    rmse = np.sqrt((error ** 2).mean())
    mape = (
        error.abs() / actual.abs().clip(lower=1)
    ).mean() * 100

    ss_res = (error ** 2).sum()
    ss_tot = ((actual - actual.mean()) ** 2).sum()

    r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0.0

    return {
        'mae': round(float(mae), 4),
        'rmse': round(float(rmse), 4),
        'mape': round(float(mape), 4),
        'r2': round(float(r2), 4),
    }


def build_prophet_model(
    regressors,
    params=None,
    use_prophet_holidays=False,
):
    if params is None:
        params = {}

    model = Prophet(
        daily_seasonality=False,
        weekly_seasonality=False,
        yearly_seasonality=False,
        seasonality_mode=params.get('seasonality_mode', 'additive'),
        changepoint_prior_scale=params.get('changepoint_prior_scale', 0.10),
        seasonality_prior_scale=params.get('seasonality_prior_scale', 10.0),
        holidays_prior_scale=params.get('holidays_prior_scale', 10.0),
    )

    model.add_seasonality(
        name='daily',
        period=1,
        fourier_order=params.get('daily_fourier_order', 16),
    )

    model.add_seasonality(
        name='weekly',
        period=7,
        fourier_order=params.get('weekly_fourier_order', 10),
    )

    model.add_seasonality(
        name='yearly',
        period=365.25,
        fourier_order=params.get('yearly_fourier_order', 12),
    )

    # Explicit, not automatic.
    if use_prophet_holidays:
        model.add_country_holidays(country_name='UK')

    for regressor in regressors:
        model.add_regressor(regressor)

    return model


## 6. Cross-validation setup

The four validation months cover different seasons and all end before the June holdout.


In [ ]:
screening_folds = [
    ('aug_2025', '2025-08-01 00:00:00', '2025-08-31 23:00:00'),
    ('nov_2025', '2025-11-01 00:00:00', '2025-11-30 23:00:00'),
    ('feb_2026', '2026-02-01 00:00:00', '2026-02-28 23:00:00'),
    ('may_2026', '2026-05-01 00:00:00', '2026-05-31 23:00:00'),
]

for fold_name, valid_start, valid_end in screening_folds:
    valid_start = pd.Timestamp(valid_start)
    valid_end = pd.Timestamp(valid_end)

    assert valid_end < FINAL_TEST_START, (
        f'{fold_name} overlaps the final test set.'
    )

    assert valid_start <= valid_end

print('All cross-validation folds safely end before June 2026.')


In [ ]:
weather_features = [
    'apparent_temperature',
    'relative_humidity_2m',
    'precipitation',
    'surface_pressure',
    'cloud_cover',
    'wind_speed_10m',
    'wind_direction_10m',
    'shortwave_radiation',
]

calendar_features = [
    'weekend',
    'is_holiday',
    'cal_event_count',
    'cal_is_covid_lockdown',
    'cal_is_general_election',
    'cal_is_major_football',
    'cal_is_non_working_day',
]

economic_features = [
    'econ_industrial_production_index_lag1m',
    'econ_gdp_index_lag1m',
    'econ_cpi_index_lag1m',
    'econ_unemployment_rate_lag1m',
]

lag_features = [
    'demand_lag_24',
    'demand_lag_168',
]

# Keep only features that actually exist.
weather_features = [c for c in weather_features if c in df.columns]
calendar_features = [c for c in calendar_features if c in df.columns]
economic_features = [c for c in economic_features if c in df.columns]

base_params = {
    'daily_fourier_order': 16,
    'weekly_fourier_order': 10,
    'yearly_fourier_order': 12,
    'changepoint_prior_scale': 0.10,
    'seasonality_prior_scale': 10.0,
    'seasonality_mode': 'additive',
}

print('Weather:', weather_features)
print('Calendar:', calendar_features)
print('Economy:', economic_features)
print('Lags:', lag_features)


In [ ]:
def evaluate_cv(
    model_name,
    regressors,
    params=None,
    use_prophet_holidays=False,
):
    if params is None:
        params = base_params

    results = []

    for fold_name, valid_start, valid_end in screening_folds:
        valid_start = pd.Timestamp(valid_start)
        valid_end = pd.Timestamp(valid_end)

        fold_train = dev_df[
            dev_df['timestamp'] < valid_start
        ].copy()

        fold_valid = dev_df[
            (dev_df['timestamp'] >= valid_start)
            & (dev_df['timestamp'] <= valid_end)
        ].copy()

        train = make_prophet_frame(fold_train, regressors)
        valid = make_prophet_frame(fold_valid, regressors)

        if train.empty or valid.empty:
            raise ValueError(
                f'{model_name} / {fold_name}: empty train or validation frame.'
            )

        model = build_prophet_model(
            regressors=regressors,
            params=params,
            use_prophet_holidays=use_prophet_holidays,
        )

        print(
            f'\n{model_name} | {fold_name} | '
            f'{len(regressors)} regressors | '
            f'train={len(train)} valid={len(valid)}'
        )

        model.fit(train[['ds', 'y', *regressors]])

        forecast = model.predict(
            valid[['ds', *regressors]]
        )

        predictions = (
            valid[['ds', 'y']]
            .merge(
                forecast[['ds', 'yhat']],
                on='ds',
                how='inner',
            )
        )

        metrics = calculate_metrics(
            predictions['y'],
            predictions['yhat'],
        )

        results.append({
            'model': model_name,
            'fold': fold_name,
            'regressor_count': len(regressors),
            'prophet_uk_holidays': use_prophet_holidays,
            **metrics,
        })

        print(metrics)

    return results


## 7. Corrected feature screening

The important holiday comparisons are now genuinely different:

- `weather_only` — no Prophet holidays
- `weather + prophet_uk_holidays` — Prophet's built-in UK holidays
- `weather + is_holiday` — your custom binary holiday feature, without Prophet UK holidays

Demand lags are also screened because historical load is relevant to short-term forecasting.


In [ ]:
screening_variants = {
    # Pure Prophet temporal structure
    'seasonality_only': {
        'regressors': [],
        'use_prophet_holidays': False,
    },

    'prophet_uk_holidays': {
        'regressors': [],
        'use_prophet_holidays': True,
    },

    # Weather
    'weather_only': {
        'regressors': weather_features,
        'use_prophet_holidays': False,
    },

    'weather + prophet_uk_holidays': {
        'regressors': weather_features,
        'use_prophet_holidays': True,
    },

    # Historical demand
    'weather + lag24': {
        'regressors': weather_features + ['demand_lag_24'],
        'use_prophet_holidays': False,
    },

    'weather + lag168': {
        'regressors': weather_features + ['demand_lag_168'],
        'use_prophet_holidays': False,
    },

    'weather + lag24 + lag168': {
        'regressors': weather_features + [
            'demand_lag_24',
            'demand_lag_168',
        ],
        'use_prophet_holidays': False,
    },

    'weather + lags + is_holiday': {
        'regressors': weather_features + [
            'demand_lag_24',
            'demand_lag_168',
            'is_holiday',
        ],
        'use_prophet_holidays': False,
    },

    'weather + lags + prophet_uk_holidays': {
        'regressors': weather_features + [
            'demand_lag_24',
            'demand_lag_168',
        ],
        'use_prophet_holidays': True,
    },
}

# Add individual calendar candidates.
for feature in calendar_features:
    screening_variants[f'weather + {feature}'] = {
        'regressors': weather_features + [feature],
        'use_prophet_holidays': False,
    }

# Add individual economic candidates.
for feature in economic_features:
    screening_variants[f'weather + {feature}'] = {
        'regressors': weather_features + [feature],
        'use_prophet_holidays': False,
    }

print('Number of screening variants:', len(screening_variants))
list(screening_variants.keys())


In [ ]:
screening_results = []

for model_name, config in screening_variants.items():
    model_results = evaluate_cv(
        model_name=model_name,
        regressors=config['regressors'],
        params=base_params,
        use_prophet_holidays=config['use_prophet_holidays'],
    )

    screening_results.extend(model_results)


In [ ]:
screening_df = pd.DataFrame(screening_results)

screening_summary = (
    screening_df
    .groupby('model')
    .agg(
        mean_mae=('mae', 'mean'),
        std_mae=('mae', 'std'),
        mean_rmse=('rmse', 'mean'),
        std_rmse=('rmse', 'std'),
        mean_mape=('mape', 'mean'),
        std_mape=('mape', 'std'),
        mean_r2=('r2', 'mean'),
        std_r2=('r2', 'std'),
    )
    .sort_values('mean_rmse')
    .reset_index()
)

screening_df.to_csv(
    OUTPUT_FOLDER / 'prophet_feature_screening.csv',
    index=False,
)

screening_summary.to_csv(
    OUTPUT_FOLDER / 'prophet_feature_summary.csv',
    index=False,
)

screening_summary


## 8. Seasonal-naive baselines

These baselines answer an important question: does Prophet beat simply using demand from the same hour yesterday or last week?

Because this project targets rolling 24-hour forecasts, lagged observations from previous periods are available at each forecast origin.


In [ ]:
naive_results = []

for fold_name, valid_start, valid_end in screening_folds:
    valid_start = pd.Timestamp(valid_start)
    valid_end = pd.Timestamp(valid_end)

    fold_valid = dev_df[
        (dev_df['timestamp'] >= valid_start)
        & (dev_df['timestamp'] <= valid_end)
    ].copy()

    baseline_configs = {
        'naive_24h': 'demand_lag_24',
        'naive_168h': 'demand_lag_168',
    }

    for baseline_name, lag_column in baseline_configs.items():
        temp = fold_valid[['demand_mw', lag_column]].dropna()

        metrics = calculate_metrics(
            temp['demand_mw'],
            temp[lag_column],
        )

        naive_results.append({
            'model': baseline_name,
            'fold': fold_name,
            **metrics,
        })

naive_df = pd.DataFrame(naive_results)

naive_summary = (
    naive_df
    .groupby('model')
    .agg(
        mean_mae=('mae', 'mean'),
        mean_rmse=('rmse', 'mean'),
        mean_mape=('mape', 'mean'),
        mean_r2=('r2', 'mean'),
    )
    .sort_values('mean_rmse')
    .reset_index()
)

naive_df.to_csv(
    OUTPUT_FOLDER / 'naive_baseline_cv.csv',
    index=False,
)

naive_summary


## 9. Small Prophet hyperparameter search

Only the two best screened Prophet feature variants are tuned. This keeps the search controlled and avoids turning the final test set into a tuning set.


In [ ]:
top_variant_names = (
    screening_summary
    .head(2)['model']
    .tolist()
)

print('Variants selected for tuning:')
top_variant_names


In [ ]:
tuning_grid = {
    'changepoint_prior_scale': [0.05, 0.10],
    'seasonality_prior_scale': [5.0, 10.0],
    'seasonality_mode': ['additive', 'multiplicative'],
}

tuning_summary_rows = []
tuning_fold_rows = []

for model_name in top_variant_names:
    config = screening_variants[model_name]

    for (
        changepoint_prior_scale,
        seasonality_prior_scale,
        seasonality_mode,
    ) in product(
        tuning_grid['changepoint_prior_scale'],
        tuning_grid['seasonality_prior_scale'],
        tuning_grid['seasonality_mode'],
    ):
        params = {
            **base_params,
            'changepoint_prior_scale': changepoint_prior_scale,
            'seasonality_prior_scale': seasonality_prior_scale,
            'seasonality_mode': seasonality_mode,
        }

        config_id = (
            f'{model_name}'
            f'_cp{changepoint_prior_scale}'
            f'_sp{seasonality_prior_scale}'
            f'_{seasonality_mode}'
        )

        fold_results = evaluate_cv(
            model_name=config_id,
            regressors=config['regressors'],
            params=params,
            use_prophet_holidays=config['use_prophet_holidays'],
        )

        fold_df = pd.DataFrame(fold_results)

        for row in fold_results:
            tuning_fold_rows.append({
                'config_id': config_id,
                'variant': model_name,
                'changepoint_prior_scale': changepoint_prior_scale,
                'seasonality_prior_scale': seasonality_prior_scale,
                'seasonality_mode': seasonality_mode,
                **row,
            })

        tuning_summary_rows.append({
            'config_id': config_id,
            'variant': model_name,
            'changepoint_prior_scale': changepoint_prior_scale,
            'seasonality_prior_scale': seasonality_prior_scale,
            'seasonality_mode': seasonality_mode,
            'mean_mae': fold_df['mae'].mean(),
            'mean_rmse': fold_df['rmse'].mean(),
            'mean_mape': fold_df['mape'].mean(),
            'mean_r2': fold_df['r2'].mean(),
            'std_rmse': fold_df['rmse'].std(),
        })


In [ ]:
tuning_summary = (
    pd.DataFrame(tuning_summary_rows)
    .sort_values('mean_rmse')
    .reset_index(drop=True)
)

tuning_fold_df = pd.DataFrame(tuning_fold_rows)

tuning_summary.to_csv(
    OUTPUT_FOLDER / 'prophet_tuning_summary.csv',
    index=False,
)

tuning_fold_df.to_csv(
    OUTPUT_FOLDER / 'prophet_tuning_folds.csv',
    index=False,
)

tuning_summary


## 10. Save the best Prophet configuration

This configuration is the Prophet candidate that should be compared with XGBoost / other models using the same development folds.

**Do not use June to choose between models.**


In [ ]:
best_row = tuning_summary.iloc[0]

best_variant_name = best_row['variant']
best_variant = screening_variants[best_variant_name]

best_params = {
    **base_params,
    'changepoint_prior_scale': float(
        best_row['changepoint_prior_scale']
    ),
    'seasonality_prior_scale': float(
        best_row['seasonality_prior_scale']
    ),
    'seasonality_mode': best_row['seasonality_mode'],
}

best_prophet_config = {
    'model_type': 'Prophet',
    'variant': best_variant_name,
    'regressors': best_variant['regressors'],
    'use_prophet_holidays': bool(
        best_variant['use_prophet_holidays']
    ),
    'params': best_params,
    'selection_metric': 'mean_cv_rmse',
    'mean_cv_rmse': float(best_row['mean_rmse']),
    'mean_cv_mae': float(best_row['mean_mae']),
    'mean_cv_mape': float(best_row['mean_mape']),
    'mean_cv_r2': float(best_row['mean_r2']),
    'final_test_start': str(FINAL_TEST_START),
}

with open(
    OUTPUT_FOLDER / 'best_prophet_config.json',
    'w',
    encoding='utf-8',
) as f:
    json.dump(
        best_prophet_config,
        f,
        indent=2,
    )

best_prophet_config


---

# STOP HERE BEFORE FINAL TEST

At this point:

1. Save the Prophet CV/tuning outputs.
2. Train and tune XGBoost (and any other candidate models) using the **same development period and folds**.
3. Compare the models using development CV metrics.
4. Select the overall winner.
5. Only then unlock June.

The cells below are deliberately disabled by default.


## 11. Final June 2026 evaluation — run once only

If Prophet is selected as the overall winning model, change `RUN_FINAL_JUNE_TEST` to `True`.

The model is trained through May 31. June is then evaluated as a rolling next-24-hour operational period. If demand-lag regressors are selected, demand from 24/168 hours earlier is information that would already be available at each daily forecast origin.


In [ ]:
RUN_FINAL_JUNE_TEST = False

if RUN_FINAL_JUNE_TEST:
    regressors = best_prophet_config['regressors']
    params = best_prophet_config['params']
    use_prophet_holidays = best_prophet_config[
        'use_prophet_holidays'
    ]

    train = make_prophet_frame(
        dev_df,
        regressors,
    )

    test = make_prophet_frame(
        final_test_df,
        regressors,
    )

    final_model = build_prophet_model(
        regressors=regressors,
        params=params,
        use_prophet_holidays=use_prophet_holidays,
    )

    print('Training final evaluation model through May 31, 2026...')

    final_model.fit(
        train[['ds', 'y', *regressors]]
    )

    forecast = final_model.predict(
        test[['ds', *regressors]]
    )

    final_predictions = (
        test[['ds', 'y']]
        .merge(
            forecast[
                [
                    'ds',
                    'yhat',
                    'yhat_lower',
                    'yhat_upper',
                ]
            ],
            on='ds',
            how='inner',
        )
    )

    final_test_metrics = calculate_metrics(
        final_predictions['y'],
        final_predictions['yhat'],
    )

    final_predictions.to_csv(
        OUTPUT_FOLDER / 'final_june_2026_predictions.csv',
        index=False,
    )

    with open(
        OUTPUT_FOLDER / 'final_june_2026_metrics.json',
        'w',
        encoding='utf-8',
    ) as f:
        json.dump(
            final_test_metrics,
            f,
            indent=2,
        )

    print('FINAL JUNE TEST METRICS:')
    print(final_test_metrics)

else:
    print(
        'June test remains LOCKED. '
        'Compare Prophet with the other models first.'
    )


## 12. Production retraining

After the final model evaluation is complete, the deployable model can be retrained with all available data, including June.

This production copy is **not** used to report June test performance.


In [ ]:
RUN_PRODUCTION_RETRAIN = False

if RUN_PRODUCTION_RETRAIN:
    regressors = best_prophet_config['regressors']
    params = best_prophet_config['params']
    use_prophet_holidays = best_prophet_config[
        'use_prophet_holidays'
    ]

    production_train = make_prophet_frame(
        df,
        regressors,
    )

    production_model = build_prophet_model(
        regressors=regressors,
        params=params,
        use_prophet_holidays=use_prophet_holidays,
    )

    production_model.fit(
        production_train[['ds', 'y', *regressors]]
    )

    with open(
        OUTPUT_FOLDER / 'prophet_model.json',
        'w',
        encoding='utf-8',
    ) as f:
        f.write(
            model_to_json(production_model)
        )

    production_metadata = {
        **best_prophet_config,
        'production_model': True,
        'trained_until': str(df['timestamp'].max()),
        'training_rows': len(production_train),
        'forecast_horizon_hours': 24,
    }

    with open(
        OUTPUT_FOLDER / 'prophet_model_metadata.json',
        'w',
        encoding='utf-8',
    ) as f:
        json.dump(
            production_metadata,
            f,
            indent=2,
        )

    print('Production Prophet model saved.')
    print('Trained until:', df['timestamp'].max())

else:
    print('Production retraining is disabled.')


## 13. Helper for future 24-hour predictions

If demand lags are among the selected regressors, create them from the historical demand table. Do **not** generate them by shifting an empty future dataframe.


In [ ]:
def attach_demand_lags(
    future_df,
    historical_df,
):
    future = future_df.copy()

    history_lookup = (
        historical_df
        .drop_duplicates(
            subset=['timestamp'],
            keep='last',
        )
        .set_index('timestamp')['demand_mw']
    )

    future['demand_lag_24'] = (
        future['timestamp']
        - pd.Timedelta(hours=24)
    ).map(history_lookup)

    future['demand_lag_168'] = (
        future['timestamp']
        - pd.Timedelta(hours=168)
    ).map(history_lookup)

    return future


# Example:
#
# future_24h = attach_demand_lags(
#     future_24h,
#     df,
# )
#
# Then ensure future_24h also contains every selected weather/calendar
# regressor before calling production_model.predict(...).
